In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path

import anndata as ad
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import c
# lr_transformation
# from pixelator.analysis.normalization import dsb_normalize


import tempfile

from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import plot_latent, plot_gene_heatmap, plot_model_latents
from doublet_seperation import B_CD4_logfc_dict, B_CD8_logfc_dict, run_cellwise_coloc_analysis_to_disk, concat_abundance_to_adata, add_doublets_metadata
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
from pixelator import read_pna as read
import pickle
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
import anndata
import hotspot

import numpy as np
import pandas as pd
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import torch
import os
from torch_geometric.data import Data
from tqdm import tqdm  # Progress bar
import glob
import numpy as np
import pandas as pd
from scipy.stats import zscore, pearsonr, spearmanr
from sklearn.decomposition import PCA


In [ ]:
DATA_DIR = Path("/home/projects/nyosef/zvise/PxlgnProject/Data")
ANNOTATED_ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/adata_annotated.h5ad'

files = [f for f in DATA_DIR.rglob('*.pxl') if f.is_file()]
data = read(files)
adata=sc.read_h5ad(ANNOTATED_ADATA_PATH)

In [ ]:
# SAVE_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/spatial_atlas_CD8.h5ad"

# gvae_raw_adata=sc.read_h5ad(SAVE_PATH)

In [ ]:
final_embeddings = np.load("/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/final_embeddings.npy")
adata.obsm['final_gvae_embeddings'] = final_embeddings
adata.obsm['final_gvae_embeddings'].shape

In [ ]:
attention_embeddings = np.load("/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_embeddings.npy")
adata.obsm['attention_embeddings'] = attention_embeddings
adata.obsm['attention_embeddings'].shape

# RAW EMBDDINGS

In [ ]:
ALL_MARKERS = adata.var_names.tolist()
MARKER_TO_IDX = {m: i for i, m in enumerate(ALL_MARKERS)}
ALL_MARKERS = list(MARKER_TO_IDX.keys())
BASE_PATH = "/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/"
BASE_PATH_ATTENTION='/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/GVAE/attention_nodes'
CACHE_DIR = os.path.join(BASE_PATH_ATTENTION, "features")


In [ ]:
# --- 4. MERGE (Run this when everything finishes) ---
print("\n--- Merging Cached Features ---")
all_feature_dfs = []
i=0
for marker in ALL_MARKERS:
    feature_save_path = os.path.join(CACHE_DIR, f"{marker}_features.pkl")
    if os.path.exists(feature_save_path):
        try:
            df = pd.read_pickle(feature_save_path)
            i=i+1
            if not df.empty:
                all_feature_dfs.append(df)
        except Exception:
            pass

if all_feature_dfs:
    df_spatial = pd.concat(all_feature_dfs, axis=1).fillna(0)
    df_spatial = df_spatial.sort_index()
    print(f"✅ DONE! Final Massive Spatial Atlas: {df_spatial.shape}")
else:
    print("⚠️ No valid data found in cache.")
print(i)

In [ ]:
df_spatial.head()

In [ ]:
feature_usage = df_spatial.sum(axis=0)
dead_features = feature_usage[feature_usage == 0]

print(f"Total Spatial Features: {df_spatial.shape[1]}")
print(f"Unused (Dead) Features: {len(dead_features)}")
if len(dead_features) > 0:
    print(f"⚠️ Dead features: {dead_features.index.tolist()}")

# 2. Check for Empty Cells (Cells with 0 signal)
# Since histograms sum to 1.0 (density), a sum of 0.0 means the cell had NO markers found.
cell_signal = df_spatial.sum(axis=1)
empty_cells = cell_signal[cell_signal == 0]
print(f"Cells with no spatial signal: {len(empty_cells)} / {df_spatial.shape[0]}")

# 3. Visualization: Feature Variance
# We want features to vary! If a feature is 0.5 for EVERY cell, it's useless.
plt.figure(figsize=(10, 4))
plt.hist(df_spatial.std(axis=0), bins=50, color='teal')
plt.title("Variance of Spatial Features")
plt.xlabel("Standard Deviation")
plt.ylabel("Count of Features")
plt.show()

In [ ]:
# Select a marker to test (e.g., FMC63)
marker = "CD8"
word_col = f"{marker}_W0" # Pick the first spatial word

# 1. Get Total Expression from original adata
# (Assuming your original adata is raw counts. If normalized, un-log it if needed)
if marker in adata.var_names:
    total_counts = adata[df_spatial.index, marker].X.toarray().flatten()
    
    # 2. Get Spatial Frequency
    spatial_freq = df_spatial[word_col].values
    
    # 3. Correlation
    correlation = np.corrcoef(total_counts, spatial_freq)[0, 1]
    
    print(f"Correlation between Total {marker} and {word_col}: {correlation:.4f}")
    
    # 4. Plot
    plt.figure(figsize=(6, 5))
    plt.scatter(total_counts, spatial_freq, alpha=0.3, s=5)
    plt.xlabel(f"Total {marker} Count")
    plt.ylabel(f"Frequency of {word_col}")
    plt.title(f"Structure vs. Abundance (R={correlation:.2f})")
    plt.show()
else:
    print(f"Marker {marker} not found in original adata.var_names")

In [ ]:
adata_spatial = ad.AnnData(df_spatial)
adata_spatial.obs = adata.obs.loc[adata_spatial.obs_names].copy()

# Run Standard UMAP
sc.pp.pca(adata_spatial)
sc.pp.neighbors(adata_spatial)
sc.tl.umap(adata_spatial)

# Plot
sc.pl.umap(
    adata_spatial, 
    color=['condition', 'cell_type'], 
    title=['Spatial State (Condition)', 'Spatial State (Cell Type)'], 
    wspace=0.3
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Prepare Data
X = df_spatial.values
y = adata_spatial.obs['condition'] # Ensure this column exists

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Train Simple Model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 4. Score
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Spatial Classifier Accuracy: {acc:.2%}")

# 5. What matters? (Feature Importance)
import pandas as pd
feat_importances = pd.Series(clf.feature_importances_, index=df_spatial.columns)
print("\nTop 10 Most Predictive Spatial Patterns:")
print(feat_importances.nlargest(10))

In [ ]:
df_spatial_aligned = df_spatial.reindex(adata.obs_names)
adata.obsm['GVAE_raw'] = df_spatial_aligned



In [ ]:
markers = {
    "Identity":   ['FMC63', 'CD3e', 'CD8'],
    "Adhesion":   ['CD11a', 'CD18', 'CD2', 'CD226'],
    "State":      ['CD45', 'CD45RA', 'CD43'],
    "Activation": ['CD69', 'CD25', 'CD137', 'CD28', 'CD38'],
    "Exhaustion": ['CD279', 'CD366', 'TIGIT', 'CD152', 'CD95']
}

# 1. PCA on Embedding (RAW - No Z-score applied to structure)
pca_df = pd.DataFrame(
    PCA(n_components=5).fit_transform(adata.obsm['GVAE_raw']), 
    index=adata.obs_names, columns=[f"PC{i+1}" for i in range(5)]
)

# 2. Bio Scores (Function)
# Step A: Z-score genes first so we can average them fairly
expr_z = adata.to_df().apply(zscore).fillna(0)

# Step B: Calculate Mean per group
bio_df = pd.DataFrame({
    group: expr_z[[g for g in genes if g in expr_z.columns]].mean(axis=1)
    for group, genes in markers.items()
}, index=adata.obs_names)

# Step C: Z-SCORE THE FINAL PROGRAMS (As requested)
bio_df = bio_df.apply(zscore).fillna(0)

# 3. Correlation & Heatmap
# Concat -> Correlate -> Slice
corr_mat = pd.concat([pca_df, bio_df], axis=1).corr(method='spearman').loc[pca_df.columns, bio_df.columns]

plt.figure(figsize=(8, 5))
sns.heatmap(corr_mat, center=0, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f")
plt.title("Raw GVAE Structure vs. Z-Scored Bio Programs")
plt.yticks(rotation=0) 
plt.show()

# trained model

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore
from sklearn.decomposition import PCA

# 0. Config
markers = {
    "Identity":   ['FMC63', 'CD3e', 'CD8'],
    "Adhesion":   ['CD11a', 'CD18', 'CD2', 'CD226'],
    "State":      ['CD45', 'CD45RA', 'CD43'],
    "Activation": ['CD69', 'CD25', 'CD137', 'CD28', 'CD38'],
    "Exhaustion": ['CD279', 'CD366', 'TIGIT', 'CD152', 'CD95']
}

# 1. PCA on Embedding (RAW - No Z-score applied to structure)
pca_df = pd.DataFrame(
    PCA(n_components=5).fit_transform(adata.obsm['final_gvae_embeddings']), 
    index=adata.obs_names, columns=[f"PC{i+1}" for i in range(5)]
)

# 2. Bio Scores (Function)
# Step A: Z-score genes first so we can average them fairly
expr_z = adata.to_df().apply(zscore).fillna(0)

# Step B: Calculate Mean per group
bio_df = pd.DataFrame({
    group: expr_z[[g for g in genes if g in expr_z.columns]].mean(axis=1)
    for group, genes in markers.items()
}, index=adata.obs_names)

# Step C: Z-SCORE THE FINAL PROGRAMS (As requested)
bio_df = bio_df.apply(zscore).fillna(0)

# 3. Correlation & Heatmap
# Concat -> Correlate -> Slice
corr_mat = pd.concat([pca_df, bio_df], axis=1).corr(method='pearson').loc[pca_df.columns, bio_df.columns]

plt.figure(figsize=(8, 5))
sns.heatmap(corr_mat, center=0, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f")
plt.title("Raw GVAE Structure vs. Z-Scored Bio Programs")
plt.yticks(rotation=0) 
plt.show()

In [ ]:

# 0. Config
markers = {
    "Identity":   ['FMC63', 'CD3e', 'CD8'],
    "Adhesion":   ['CD11a', 'CD18', 'CD2', 'CD226'],
    "State":      ['CD45', 'CD45RA', 'CD43'],
    "Activation": ['CD69', 'CD25', 'CD137', 'CD28', 'CD38'],
    "Exhaustion": ['CD279', 'CD366', 'TIGIT', 'CD152', 'CD95']
}

# 1. PCA on Embedding (RAW - No Z-score applied to structure)
pca_df = pd.DataFrame(adata.obsm['final_gvae_embeddings'], index=adata.obs_names, columns=[f"Dim{i+1}" for i in range(adata.obsm['final_gvae_embeddings'].shape[1])])

# 2. Bio Scores (Function)
# Step A: Z-score genes first so we can average them fairly
expr_z = adata.to_df().apply(zscore).fillna(0)

# Step B: Calculate Mean per group
bio_df = pd.DataFrame({
    group: expr_z[[g for g in genes if g in expr_z.columns]].mean(axis=1)
    for group, genes in markers.items()
}, index=adata.obs_names)

# Step C: Z-SCORE THE FINAL PROGRAMS (As requested)
bio_df = bio_df.apply(zscore).fillna(0)

# 3. Correlation & Heatmap
# Concat -> Correlate -> Slice
corr_mat = pd.concat([pca_df, bio_df], axis=1).corr(method='pearson').loc[pca_df.columns, bio_df.columns]

plt.figure(figsize=(16, 10))
sns.heatmap(corr_mat, center=0, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f")
plt.title("Raw GVAE Structure vs. Z-Scored Bio Programs")
plt.yticks(rotation=0) 
plt.show()

# ATTENTION MODEL

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore
from sklearn.decomposition import PCA

# 0. Config
markers = {
    "Identity":   ['FMC63', 'CD3e', 'CD8'],
    "Adhesion":   ['CD11a', 'CD18', 'CD2', 'CD226'],
    "State":      ['CD45', 'CD45RA', 'CD43'],
    "Activation": ['CD69', 'CD25', 'CD137', 'CD28', 'CD38'],
    "Exhaustion": ['CD279', 'CD366', 'TIGIT', 'CD152', 'CD95']
}

# 1. PCA on Embedding (RAW - No Z-score applied to structure)
pca_df = pd.DataFrame(
    PCA(n_components=5).fit_transform(adata.obsm['attention_embeddings']), 
    index=adata.obs_names, columns=[f"PC{i+1}" for i in range(5)]
)

# 2. Bio Scores (Function)
# Step A: Z-score genes first so we can average them fairly
expr_z = adata.to_df().apply(zscore).fillna(0)

# Step B: Calculate Mean per group
bio_df = pd.DataFrame({
    group: expr_z[[g for g in genes if g in expr_z.columns]].mean(axis=1)
    for group, genes in markers.items()
}, index=adata.obs_names)

# Step C: Z-SCORE THE FINAL PROGRAMS (As requested)
bio_df = bio_df.apply(zscore).fillna(0)

# 3. Correlation & Heatmap
# Concat -> Correlate -> Slice
corr_mat = pd.concat([pca_df, bio_df], axis=1).corr(method='pearson').loc[pca_df.columns, bio_df.columns]

plt.figure(figsize=(8, 5))
sns.heatmap(corr_mat, center=0, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f")
plt.title("Raw GVAE Structure vs. Z-Scored Bio Programs")
plt.yticks(rotation=0) 
plt.show()

In [ ]:

# 0. Config
markers = {
    "Identity":   ['FMC63', 'CD3e', 'CD8'],
    "Adhesion":   ['CD11a', 'CD18', 'CD2', 'CD226'],
    "State":      ['CD45', 'CD45RA', 'CD43'],
    "Activation": ['CD69', 'CD25', 'CD137', 'CD28', 'CD38'],
    "Exhaustion": ['CD279', 'CD366', 'TIGIT', 'CD152', 'CD95']
}

# 1. PCA on Embedding (RAW - No Z-score applied to structure)
pca_df = pd.DataFrame(adata.obsm['attention_embeddings'], index=adata.obs_names, columns=[f"Dim{i+1}" for i in range(adata.obsm['attention_embeddings'].shape[1])])

# 2. Bio Scores (Function)
# Step A: Z-score genes first so we can average them fairly
expr_z = adata.to_df().apply(zscore).fillna(0)

# Step B: Calculate Mean per group
bio_df = pd.DataFrame({
    group: expr_z[[g for g in genes if g in expr_z.columns]].mean(axis=1)
    for group, genes in markers.items()
}, index=adata.obs_names)

# Step C: Z-SCORE THE FINAL PROGRAMS (As requested)
bio_df = bio_df.apply(zscore).fillna(0)

# 3. Correlation & Heatmap
# Concat -> Correlate -> Slice
corr_mat = pd.concat([pca_df, bio_df], axis=1).corr(method='pearson').loc[pca_df.columns, bio_df.columns]

plt.figure(figsize=(16, 10))
sns.heatmap(corr_mat, center=0, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f")
plt.title("Raw GVAE Structure vs. Z-Scored Bio Programs")
plt.yticks(rotation=0) 
plt.show()